[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/probability_course.ipynb)

# Probability full course: practice notebook

One practice question for every teaching part of the video, in the same order, and three cells you can play with:
one builds sample spaces for as many coins as you like, one rolls two dice thousands of times, and one deals hands of
cards and checks the counting formula against thousands of deals. Work each question out on paper first, then type
your answer into the box and press **check**. It tells you whether you're right. Press **show working** if you'd like
to see the steps.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the cell runs
again by itself. Change any number and run the cell again: nothing can break. It works in your browser (no install)
and in Google Colab.

**Typing answers.** Probabilities as fractions: `5/36` (you don't have to simplify). Decimals are fine too, to at least
three places: `0.139`. Whole numbers without commas or spaces: `22100`. You can type the working: `1 - 25/36`,
`52C3` for 52 choose 3. For several answers, separate them with commas, in the order the question asks. You can also
put your answer between the quotes in a cell's last line, `practice(1, "")`, and run the cell: it checks straight
away.

**The cheat sheet.** Every rule from the course on two pages:
[probability cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/probability_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import math, re, textwrap, random
from math import comb, perm
from fractions import Fraction as F
from itertools import product, combinations
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import sympy as sp
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor
from IPython.display import display
from ipywidgets import interact, IntSlider, SelectionSlider, ToggleButtons, Text, Button, Output, HBox, VBox, Layout

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # chalk, chalk smudge, ghost lines
CORAL = "#e3896b"                           # the one key thing in each figure
SKY, MINT, GOLD = "#6fb6bb", "#a0c27c", "#e8c56b"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False, "font.size": 13, "axes.titlesize": 15, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

RULES = standard_transformations + (implicit_multiplication_application, convert_xor)
NAMES = {"binomial": sp.binomial, "ff": sp.ff, "factorial": sp.factorial}
SUPS, SUBS = "⁰¹²³⁴⁵⁶⁷⁸⁹", "₀₁₂₃₄₅₆₇₈₉"
def sup(n): return "".join(SUPS[int(d)] for d in str(n))
def sub(n): return "".join(SUBS[int(d)] for d in str(n))

def prep(s):
    """Typed symbols → Python: − × ÷, 52C3 and ⁵²C₃ (choose), 22 100 → 22100."""
    s = s.replace("−", "-").replace("×", "*").replace("÷", "/").replace("·", "*")
    s = re.sub(f"([{SUPS}]+)\\s*([Cc])\\s*([{SUBS}]+)",
               lambda m: "".join(str(SUPS.index(c)) for c in m[1]) + "C" + "".join(str(SUBS.index(c)) for c in m[3]), s)
    s = re.sub(r"(\d+)\s*[Cc]\s*(\d+)", r"binomial(\1,\2)", s)
    return re.sub(r"(?<=\d) (?=\d{3}\b)", "", s)

def read(s):
    """Your typed answer (a list, one for each answer)."""
    return [parse_expr(prep(p), local_dict=NAMES, transformations=RULES) for p in s.split(",") if p.strip()]

def same(got, want, tol=1e-9):
    """Exact for fractions and whole numbers; a typed decimal may be rounded to three places."""
    try:
        got = sp.sympify(got)
        if got.atoms(sp.Float): tol = max(tol, 0.0005 + 1e-12)
        return abs(float(sp.N(got - sp.Rational(want.numerator, want.denominator), 30))) <= tol
    except (TypeError, ValueError): return False

# Each question: the question, the answers in order, the working, and the move that cracks each part (shown when that
# part is wrong).
Q = {}

def practice(n, answer=""):
    """Question n: an answer box, a check button and a show-working button."""
    q = Q[n]
    box = Text(value=answer, placeholder="your answer", description="answer:", layout=Layout(width="460px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            try: got = read(box.value)
            except Exception: print("I can't read that yet: check the brackets and the signs."); return
            want = q["answer"]
            if len(got) != len(want): print(f"✗ the question asks for {len(want)} answer(s); you typed {len(got)}."); return
            ok = [same(g, w) for g, w in zip(got, want)]
            if all(ok): print("✓ right"); return
            for k, (good, move) in enumerate(zip(ok, q["moves"]), 1):
                print(f"✓ part {k}: right" if good else f"✗ part {k}: not quite. The move that cracks it: {move}")
    def on_work(_=None):
        out.clear_output()
        with out:
            for line in q["working"]: print(line)
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"Question {n}. {q['text']}", 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def P(event, space):
    """The classical definition, by listing: the outcomes in the event over all the outcomes."""
    space = list(space)
    return F(sum(1 for w in space if event(w)), len(space))

DICE = list(product(range(1, 7), repeat=2))                    # (first die, second die): 36 equally likely outcomes
WAYS = {t: sum(1 for o in DICE if sum(o) == t) for t in range(2, 13)}

## 1 · Random experiments and sample spaces

A random experiment has more than one possible result, and you can't say in advance which one you'll get. Each
possible result is an outcome, and the set of every outcome is the sample space S. The number of outcomes in it is
n(S): a coin has S = {H, T} and n(S) = 2; a die has S = {1, 2, 3, 4, 5, 6} and n(S) = 6. For two coins, HT and TH are
different outcomes, so n(S) = 4. With more coins, a tree diagram keeps the list honest: each coin splits every branch
in two. And for two dice, a 6 × 6 grid of pairs gives 36 outcomes.

The first cell builds sample spaces. Move the slider to choose how many coins to toss: it draws the tree and lists
every outcome, with the count underneath.

In [ ]:
def coin_tree(coins=3):
    outs = ["".join(p) for p in product("HT", repeat=coins)]     # Python lists every outcome …
    n = len(outs)                                                 # … and counts the list
    assert n == 2 ** coins
    fig = plt.figure(figsize=(11.5, 5.4))
    ax = fig.add_axes([0.01, 0.02, 0.55, 0.88]); ax.axis("off")
    m = 2 ** coins
    ys = [(m - 1) / 2 - j for j in range(m)]                      # the tips, top to bottom
    def y(level, i):                                              # a node sits level with the middle of its tips
        w = 2 ** (coins - level)
        return sum(ys[i * w:(i + 1) * w]) / w
    fs = {1: 16, 2: 15, 3: 13, 4: 10, 5: 7.5}[coins]
    for level in range(coins):
        for i in range(2 ** level):
            for k, face in enumerate("HT"):
                j = 2 * i + k
                ax.plot([level, level + 1], [y(level, i), y(level + 1, j)], color=GHOST if coins > 3 else DIM,
                        lw=1.6 if coins <= 3 else 1.0, zorder=1)
                ax.text(level + 1, y(level + 1, j), face, ha="center", va="center", fontsize=fs, zorder=3,
                        color=SKY if face == "H" else CORAL, fontweight="bold",
                        bbox=dict(boxstyle="circle,pad=0.18", fc=SLATE, ec="none"))
    ax.plot([0], [y(0, 0)], "o", color=CHALK, ms=7)
    for level in range(1, coins + 1):
        ax.text(level, (m - 1) / 2 + 0.9 + 0.02 * m, f"coin {level}", ha="center", color=DIM, fontsize=11)
    if coins <= 4:
        for j, w in enumerate(outs):
            ax.text(coins + 0.35, ys[j], w, va="center", color=CHALK, fontsize=fs, family="monospace")
    ax.set_xlim(-0.3, coins + 1.4); ax.set_ylim(-(m - 1) / 2 - 0.7, (m - 1) / 2 + 1.3 + 0.03 * m)
    fig.text(0.02, 0.94, f"a tree for {coins} coin{'s' if coins > 1 else ''}: every branch splits in two",
             color=DIM, fontsize=12)
    tp = fig.add_axes([0.58, 0.0, 0.42, 1.0]); tp.axis("off"); tp.set_xlim(0, 1); tp.set_ylim(0, 1)
    tp.text(0.0, 0.92, "the sample space S, listed", color=DIM, fontsize=13)
    rows = 8
    cols = max(1, -(-n // rows))
    for k, w in enumerate(outs):
        col, row = divmod(k, rows)
        tp.text(0.0 + col * (0.98 / max(cols, 4)), 0.83 - 0.07 * row, w + ("," if k < n - 1 else ""),
                color=CHALK, fontsize=13 if coins <= 4 else 12, family="monospace")
    tp.text(0.0, 0.18, f"Python counted the list: {n} outcomes", color=CHALK, fontsize=15)
    tp.text(0.0, 0.07, f"n(S) = 2{sup(coins)} = {n}", color=GOLD, fontsize=22, fontweight="bold")
    plt.show()

assert len(list(product("HT", repeat=2))) == 4 and len(list(product("HT", repeat=3))) == 8   # the video: 4 and 8
assert len(DICE) == 36 and len(WAYS) == 11                                                  # … 36 cells, 11 totals

In [ ]:
interact(coin_tree, coins=IntSlider(value=3, min=1, max=5, description="coins"));

In [ ]:
FAMILY = list(product("BG", repeat=3))                                   # three children, in birth order
SPIN2 = list(product(["red", "blue", "green", "yellow"], repeat=2))      # a spinner with four colours, spun twice
GAME = [("H", k) for k in range(1, 7)] + [("T", c) for c in ["red", "blue", "green"]]
Q[1] = {"text": "A family in Lima has three children. Write each child as B or G, in birth order (BBG, BGB, …). How "
                "many outcomes are in the sample space? Then: a spinner with four equal colours is spun twice. How "
                "many outcomes? And at a game night in Seoul, you toss a coin: heads, you roll a die; tails, you spin "
                "a spinner with three colours. How many outcomes are there now?",
        "answer": [len(FAMILY), len(SPIN2), len(GAME)],
        "moves": ["each child splits every branch in two: 2 × 2 × 2.",
                  "four colours for the first spin, and each meets four for the second: 4 × 4.",
                  "the branches split differently, so don't multiply: count the tips, 6 after heads and 3 after tails."],
        "working": ["Three children: a tree with 2 branches, then 2, then 2, so 2 × 2 × 2 = 8 outcomes:",
                    "BBB, BBG, BGB, BGG, GBB, GBG, GGB, GGG.",
                    "The spinner twice: 4 colours, then 4 again, so 4 × 4 = 16 ordered pairs (red-blue isn't blue-red).",
                    "The game: heads leads to 6 outcomes (H1 to H6), tails to 3 (T-red, T-blue, T-green).",
                    "S = {H1, H2, H3, H4, H5, H6, T-red, T-blue, T-green}, so n(S) = 6 + 3 = 9."]}
assert Q[1]["answer"] == [2 ** 3, 4 * 4, 6 + 3] == [8, 16, 9]
practice(1)

## 2 · Events

An event is a set of outcomes: a subset of the sample space. It happens when the outcome you get is in it. A simple
event has one outcome; a compound event has more. The sure event is S itself, and the impossible event is the empty
set ∅. Events combine like sets: "not A" is everything outside A, "A or B" (A ∪ B) is everything in either, "A and B"
(A ∩ B) is everything in both, and "A but not B" is A with B taken out. Two events are mutually exclusive when they
share no outcome, and exhaustive when together they cover all of S.

In [ ]:
TICKETS = set(range(1, 13))
A = {k for k in TICKETS if k % 3 == 0}                                   # a multiple of 3
B = {k for k in TICKETS if k % 2 == 1}                                   # an odd number
Q[2] = {"text": "A raffle at a school in Cape Town has tickets numbered 1 to 12, and one is drawn. Event A is 'a "
                "multiple of 3' and event B is 'an odd number'. How many outcomes are in A or B? How many are in A "
                "and B? And how many are in neither A nor B?",
        "answer": [len(A | B), len(A & B), len(TICKETS - (A | B))],
        "moves": ["list both: A = {3, 6, 9, 12}, B = {1, 3, 5, 7, 9, 11}; 'or' takes everything in either, once.",
                  "'and' keeps only the numbers in both lists.",
                  "neither = everything outside A or B: 12 minus your first answer."],
        "working": ["A = {3, 6, 9, 12} and B = {1, 3, 5, 7, 9, 11}.",
                    "A or B = {1, 3, 5, 6, 7, 9, 11, 12}: 8 outcomes (3 and 9 are in both, but count once).",
                    "A and B = {3, 9}: 2 outcomes. Check: 4 + 6 − 2 = 8.",
                    "Neither: {2, 4, 8, 10}, so 12 − 8 = 4 outcomes."]}
assert Q[2]["answer"] == [8, 2, 4] and len(A) + len(B) - len(A & B) == 8
practice(2)

## 3 · The classical definition of probability

When every outcome is equally likely, the probability of an event E is P(E) = n(E) / n(S): the outcomes in the event
over all the outcomes. It's always between 0 (impossible) and 1 (certain). For two dice, the 36 cells of the grid are
the equally likely outcomes, not the 11 totals: a total of 7 has six cells, the most of any total, so P(7) = 6/36 =
1/6. There's another way to measure chance, too: try it many times and divide the number of times the event happened
by the number of trials. That's the experimental probability, or relative frequency, and in the long run it settles
down near the true probability.

You can watch this happen in the cell below. It rolls two dice as many times as you choose with its slider, and draws
the share of each total as bars next to the exact triangle from the grid. It also prints the share of sevens, so you
can see it closing in on one sixth. The rolls come from a fixed seed, so you'll see the same numbers as the video.

In [ ]:
SIM = random.Random(2026)                                                # the video's simulator: the same seed
ROLLS = [(int(SIM.random() * 6) + 1, int(SIM.random() * 6) + 1) for _ in range(10000)]
SEVEN = np.cumsum([sum(o) == 7 for o in ROLLS])                          # sevens so far, after each roll
SEV = {n: int(SEVEN[n - 1]) for n in (10, 100, 1000, 10000)}
assert SEV == {10: 0, 100: 13, 1000: 146, 10000: 1699}                   # the video: 0, 13, 146, 1699 sevens
assert max(WAYS, key=WAYS.get) == 7 and F(WAYS[7], 36) == F(1, 6)

def two_dice(rolls=10):
    got = ROLLS[:rolls]
    count = {t: sum(1 for o in got if sum(o) == t) for t in range(2, 13)}
    share = {t: count[t] / rolls for t in count}
    fig = plt.figure(figsize=(11.5, 4.9))
    ax = fig.add_axes([0.06, 0.13, 0.5, 0.72])
    T = np.arange(2, 13)
    ax.bar(T - 0.2, [share[t] for t in T], width=0.4, color=[CORAL if t == 7 else SKY for t in T], label="your rolls")
    ax.bar(T + 0.2, [WAYS[t] / 36 for t in T], width=0.4, fc="none", ec=GOLD, lw=1.6, label="exact: cells ÷ 36")
    ax.plot(T + 0.2, [WAYS[t] / 36 for t in T], color=GOLD, lw=1.2, ls=":")
    ax.set_xticks(T); ax.set_ylim(0, 0.4); ax.set_xlabel("total of the two dice")
    ax.set_ylabel("share of the rolls")
    ax.legend(loc="upper right", frameon=False)
    ax.set_title(f"{rolls} rolls of two dice", loc="left", color=DIM, fontsize=13)
    bx = fig.add_axes([0.66, 0.13, 0.31, 0.46])
    n = np.arange(1, rolls + 1)
    bx.plot(n, SEVEN[:rolls] / n, color=CORAL, lw=1.6)
    bx.axhline(1 / 6, color=GOLD, ls="--", lw=1.4)
    bx.text(1.3, 1 / 6 + 0.02, "1/6 ≈ 0.167", color=GOLD, fontsize=11)
    bx.set_xscale("log"); bx.set_xlim(1, 10000); bx.set_ylim(0, 0.5)
    bx.set_xticks([1, 10, 100, 1000, 10000]); bx.set_xticklabels(["1", "10", "100", "1000", "10000"])
    bx.minorticks_off()
    bx.set_xlabel("rolls so far"); bx.set_title("the share of sevens", loc="left", color=DIM, fontsize=12)
    s7 = int(SEVEN[rolls - 1])
    fig.text(0.66, 0.9, f"sevens: {s7} of {rolls}", color=CHALK, fontsize=16)
    fig.text(0.66, 0.8, f"share = {s7 / rolls:.3f}", color=CORAL, fontsize=17, fontweight="bold")
    fig.text(0.66, 0.72, "exact: 1/6 ≈ 0.167", color=GOLD, fontsize=13)
    plt.show()

In [ ]:
interact(two_dice, rolls=SelectionSlider(options=[10, 20, 50, 100, 200, 500, 1000, 2000, 5000, 10000], value=10,
                                         description="rolls"));

In [ ]:
WORD = "BANANA"
Q[3] = {"text": "A board game in Berlin uses two dice. What is the probability that the total is 8? What is the "
                "probability that the two numbers multiply to make 12? And the six letters of BANANA are written on "
                "tiles and one tile is picked at random: what is the probability that it shows N?",
        "answer": [P(lambda o: sum(o) == 8, DICE), P(lambda o: o[0] * o[1] == 12, DICE), P(lambda c: c == "N", WORD)],
        "moves": ["count cells on the 6 × 6 grid, not totals: (2, 6), (3, 5), … out of 36.",
                  "list the pairs that multiply to 12, in both orders, out of 36.",
                  "the tiles are the equally likely outcomes: count N tiles out of 6."],
        "working": ["Total 8: (2, 6), (3, 5), (4, 4), (5, 3), (6, 2), five cells, so P = 5/36.",
                    "Product 12: (2, 6), (3, 4), (4, 3), (6, 2), four cells, so P = 4/36 = 1/9.",
                    "BANANA has 6 tiles, two of them N, so P(N) = 2/6 = 1/3."]}
assert Q[3]["answer"] == [F(5, 36), F(1, 9), F(1, 3)] and WAYS[8] == 5
practice(3)

## 4 · The axiomatic approach to probability

When outcomes aren't equally likely (a weighted die, a spinner with unequal parts), give each outcome its own
probability, following three rules, the axioms: a probability is never negative; P(S) = 1, so the probabilities of
all the outcomes add up to one; and for events that can't happen together, the probabilities add. The probability of
an event is the sum of the probabilities of its outcomes. From the axioms come P(∅) = 0 and 0 ≤ P(E) ≤ 1, and the
classical definition is the special case where all n outcomes get 1/n each.

In [ ]:
SPIN = {"red": F(3, 10), "blue": F(1, 4)}
GREEN = (1 - sum(SPIN.values())) / 2                                     # green and yellow share what's left equally
ODD = F(1, 3 * 1 + 3 * 2)                                                # each odd face p, each even face 2p: 9p = 1
LOADED = {k: ODD * (2 if k % 2 == 0 else 1) for k in range(1, 7)}
assert sum(LOADED.values()) == 1 and all(p >= 0 for p in LOADED.values())
Q[4] = {"text": "A spinner at a school fair in Manila has four colours: P(red) = 0.3, P(blue) = 0.25, and green and "
                "yellow are equally likely. What is P(green)? Then a weighted die: each even number is twice as likely "
                "as each odd number. What is P(6)? And what is the probability of rolling 3 or less?",
        "answer": [GREEN, LOADED[6], LOADED[1] + LOADED[2] + LOADED[3]],
        "moves": ["the four probabilities add up to 1: green + yellow = 1 − 0.3 − 0.25, shared equally.",
                  "call each odd face p and each even face 2p; the six add up to 1, so 3p + 6p = 1.",
                  "3 or less means 1, 2 or 3: add their probabilities, p + 2p + p."],
        "working": ["Spinner: 0.3 + 0.25 = 0.55, so green and yellow share 1 − 0.55 = 0.45.",
                    "Equally, so P(green) = 0.45 ÷ 2 = 0.225 (that's 9/40).",
                    "Weighted die: odd faces p, even faces 2p. 3p + 3 × 2p = 9p = 1, so p = 1/9.",
                    "P(6) = 2p = 2/9.",
                    "P(3 or less) = P(1) + P(2) + P(3) = 1/9 + 2/9 + 1/9 = 4/9."]}
assert Q[4]["answer"] == [F(9, 40), F(2, 9), F(4, 9)] and float(GREEN) == 0.225
practice(4)

## 5 · The addition rule and the complement

For any two events, P(A or B) = P(A) + P(B) − P(A and B): adding P(A) and P(B) counts the overlap twice, so take it
away once. If A and B are mutually exclusive, there's no overlap, and P(A or B) = P(A) + P(B). "A" and "not A" share
nothing and cover everything, so P(not A) = 1 − P(A): often the quickest way to "at least one". Together with a Venn
diagram, the rules give every region: neither, exactly one, A but not B.

In [ ]:
CLUB = 40
SAT, SUN, BOTH = 22, 15, 9                                               # members who run on Saturday, Sunday, both
EITHER = F(SAT, CLUB) + F(SUN, CLUB) - F(BOTH, CLUB)
Q[5] = {"text": "A running club in Nairobi has 40 members: 22 run on Saturdays, 15 on Sundays, and 9 on both days. One "
                "member is picked at random. What is the probability they run on Saturday or Sunday? That they run on "
                "neither day? And that they run on exactly one of the two days?",
        "answer": [EITHER, 1 - EITHER, F((SAT - BOTH) + (SUN - BOTH), CLUB)],
        "moves": ["the addition rule: 22/40 + 15/40 − 9/40, because the 9 were counted twice.",
                  "neither is the complement of 'Saturday or Sunday': 1 minus your first answer.",
                  "exactly one = Saturday only + Sunday only: (22 − 9) + (15 − 9), out of 40."],
        "working": ["P(Sat or Sun) = 22/40 + 15/40 − 9/40 = 28/40 = 7/10.",
                    "P(neither) = 1 − 7/10 = 3/10 (12 members).",
                    "Saturday only: 22 − 9 = 13. Sunday only: 15 − 9 = 6.",
                    "P(exactly one) = (13 + 6)/40 = 19/40. Check: 13 + 9 + 6 + 12 = 40."]}
assert Q[5]["answer"] == [F(7, 10), F(3, 10), F(19, 40)] and 13 + 9 + 6 + 12 == CLUB
practice(5)

## 6 · Probability by counting

When the outcomes are too many to list, count them. If one thing can happen in m ways and then another in n ways, the
two together happen in m × n ways. n different things can be put in a row in n! ways, and r of them in ⁿPᵣ = n!/(n − r)!
ways. Choosing r of them, where order doesn't matter, can be done in ⁿCᵣ = n!/(r!(n − r)!) ways: "n choose r". A deck
has 52 cards, 13 in each of four suits (hearts and diamonds red, clubs and spades black), and two cards dealt together
can be any of 52 choose 2 = 1326 pairs.

The deal cell deals hands from a deck and shows the counting formula next to each hand. Pick how many cards and what
you're looking for, and move **deal** to deal a new hand. Underneath, it runs 20,000 deals, to compare the simulated
share with the exact answer.

In [ ]:
DECK = [(r, s) for r in range(1, 14) for s in "HDCS"]                    # rank 1 = ace, 11–13 = jack, queen, king
RANK = {1: "A", 11: "J", 12: "Q", 13: "K"}
SUIT = {"H": "♥", "D": "♦", "C": "♣", "S": "♠"}
EVENTS = {"all hearts": lambda h: all(s == "H" for _, s in h),
          "all aces": lambda h: all(r == 1 for r, _ in h),
          "at least one ace": lambda h: any(r == 1 for r, _ in h)}

def exact(event, k):
    """n(E) by counting, and the formula that counts it."""
    if event == "all hearts": return comb(13, k), f"¹³C{sub(k)}", f"choose {k} of the 13 hearts"
    if event == "all aces": return comb(4, k), f"⁴C{sub(k)}", f"choose {k} of the 4 aces"
    return comb(52, k) - comb(48, k), f"⁵²C{sub(k)} − ⁴⁸C{sub(k)}", "every hand minus the hands with no ace"

assert F(exact("all aces", 2)[0], comb(52, 2)) == F(1, 221)              # the video: both aces 6/1326
assert F(exact("all hearts", 2)[0], comb(52, 2)) == F(1, 17)             # … both hearts 78/1326
assert F(exact("all hearts", 5)[0], comb(52, 5)) == F(1287, 2598960)     # … five hearts, about 1 in 2000
for k in range(1, 6):                                                     # the formulas agree with listing
    if k <= 3:
        for ev in EVENTS:
            assert sum(1 for h in combinations(DECK, k) if EVENTS[ev](h)) == exact(ev, k)[0]

RK = np.array([r for r, _ in DECK]); ST = np.array([s for _, s in DECK])
DEALS, SIMS = 20000, {}
def simulate(k):
    """20,000 hands of k cards from a shuffled deck (a fixed seed, so the numbers stay put)."""
    if k not in SIMS:
        rng = np.random.default_rng(2026 + k)
        SIMS[k] = rng.random((DEALS, 52)).argpartition(k, axis=1)[:, :k]
    return SIMS[k]

def share(event, k):
    h = simulate(k)
    if event == "all hearts": hit = (ST[h] == "H").all(axis=1)
    elif event == "all aces": hit = (RK[h] == 1).all(axis=1)
    else: hit = (RK[h] == 1).any(axis=1)
    return int(hit.sum())

def draw_card(ax, x, card, lit):
    r, s = card
    red = s in "HD"
    ax.add_patch(FancyBboxPatch((x, 0), 1.0, 1.4, boxstyle="round,pad=0.02,rounding_size=0.1", fc="#f4f1ea",
                                ec=GOLD if lit else DIM, lw=3 if lit else 1.2))
    ink = "#c0503a" if red else "#1f2b28"
    ax.text(x + 0.12, 1.22, RANK.get(r, str(r)), color=ink, fontsize=15, fontweight="bold", va="center")
    ax.text(x + 0.5, 0.62, SUIT[s], color=ink, fontsize=34, ha="center", va="center")

def deal(cards=2, look_for="all hearts", deal=1):
    hand = random.Random(100 * cards + deal).sample(DECK, cards)          # deal number n is always the same hand
    fits = EVENTS[look_for](hand)
    nE, formula, why = exact(look_for, cards)
    nS = comb(52, cards)
    p = F(nE, nS)
    hits = share(look_for, cards)
    fig = plt.figure(figsize=(11.5, 5.0))
    ax = fig.add_axes([0.02, 0.42, 0.46, 0.42]); ax.axis("off"); ax.set_aspect("equal")
    for i, c in enumerate(hand): draw_card(ax, i * 1.15, c, fits)
    ax.set_xlim(-0.1, 5 * 1.15); ax.set_ylim(-0.1, 1.55)
    fig.text(0.02, 0.92, f"deal {deal}: {cards} cards", color=DIM, fontsize=13)
    fig.text(0.02, 0.32, f"'{look_for}'?  " + ("✓ yes, this hand fits" if fits else "✗ not this hand"),
             color=GOLD if fits else CHALK, fontsize=16, fontweight="bold" if fits else "normal")
    tp = fig.add_axes([0.5, 0.0, 0.5, 1.0]); tp.axis("off"); tp.set_xlim(0, 1); tp.set_ylim(0, 1)
    tp.text(0.0, 0.9, "counting", color=DIM, fontsize=13)
    tp.text(0.0, 0.8, f"every hand:  n(S) = ⁵²C{sub(cards)} = {nS:,}", color=CHALK, fontsize=15)
    tp.text(0.0, 0.7, f"'{look_for}':  n(E) = {formula} = {nE:,}", color=CHALK, fontsize=15)
    tp.text(0.0, 0.63, f"({why})", color=DIM, fontsize=11.5)
    frac = f"{nE:,}/{nS:,}" + (f" = {p.numerator}/{p.denominator}" if p.denominator != nS and p else "")
    tp.text(0.0, 0.5, f"P = {frac}", color=CORAL, fontsize=16 if len(frac) < 26 else 13, fontweight="bold")
    tp.text(0.0, 0.42, f"   ≈ {float(p):.4f}" if float(p) >= 0.001 else f"   ≈ {float(p):.6f}", color=CORAL, fontsize=15)
    tp.text(0.0, 0.26, f"{DEALS:,} simulated deals: {hits:,} fit", color=CHALK, fontsize=14)
    small = float(p) < 0.01
    tp.text(0.0, 0.17, f"simulated share ≈ {hits / DEALS:.6f}" if small else f"simulated share ≈ {hits / DEALS:.4f}",
            color=GOLD, fontsize=16, fontweight="bold")
    tp.text(0.0, 0.08, "a rare hand: the simulation is still settling, the count is exact" if small and p else
            "close to the exact answer: counting works", color=DIM, fontsize=12)
    plt.show()

In [ ]:
interact(deal, cards=IntSlider(value=2, min=1, max=5, description="cards"),
         look_for=ToggleButtons(options=list(EVENTS), description=" "),
         deal=IntSlider(value=1, min=1, max=30, description="deal"));

In [ ]:
HANDS3 = list(combinations(DECK, 3))                                     # Python lists every 3-card hand
Q[6] = {"text": "Your turn to count a hand. At a card night in Buenos Aires, you're dealt 3 cards from a shuffled deck "
                "of 52. How many different hands are there? What is the probability that all three cards are red? And "
                "what is the probability that you get at least one ace?",
        "answer": [len(HANDS3), P(lambda h: all(s in "HD" for _, s in h), HANDS3),
                   P(lambda h: any(r == 1 for r, _ in h), HANDS3)],
        "moves": ["order doesn't matter in a hand: 52 choose 3.",
                  "choose 3 of the 26 red cards, then divide by every hand: 26C3 / 52C3.",
                  "at least one: use the complement. Hands with no ace are 48C3, so 1 − 48C3/52C3."],
        "working": ["Every hand: 52C3 = (52 × 51 × 50) ÷ (3 × 2 × 1) = 22100.",
                    "All red: 26C3 = (26 × 25 × 24) ÷ 6 = 2600, so P = 2600/22100 = 2/17 ≈ 0.118.",
                    "No ace: 48C3 = (48 × 47 × 46) ÷ 6 = 17296, so P(no ace) = 17296/22100.",
                    "P(at least one ace) = 1 − 17296/22100 = 4804/22100 = 1201/5525 ≈ 0.217.",
                    "(Try it in the deal cell: 3 cards, 'at least one ace'.)"]}
assert Q[6]["answer"] == [comb(52, 3), F(comb(26, 3), comb(52, 3)), 1 - F(comb(48, 3), comb(52, 3))]
assert Q[6]["answer"] == [22100, F(2, 17), F(1201, 5525)]
practice(6)

## Sources
NCERT Mathematics Class XI, ch. 14 (Probability); OpenStax, Precalculus 2e, 13.5 (Counting Principles) and 13.7
(Probability). Both retrieved 2026-11-01. Every question is our own.